In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

import xclimate as xclim

In [ ]:
## FHIST PPE

variables = [
    'EFLX_LH_TOT_month_1',
    'TLAI_month_1',
    'PRECC_month_1',
    'PRECL_month_1',
]

time_slice = slice('1950-01', '2014-12')
year_start = time_slice.start[:4]
year_end = time_slice.stop[:4]
grid = xclim.load_fhist_ppe_grid()

snow_pct_threshold = 100    # maximum allowable percent of snow cover on all months of the average year, 100 = no mask
nonglc_pct_threshold = 100  # maximum allowable percent of glaciated land for a NON-glaciated gridcell, 100 = no mask

fsno = xclim.load_fhist('FSNO_month_1', keep_var_only=True)['FSNO'].sel(time=time_slice).reindex_like(grid, method='nearest', tolerance=1e-3)
fsno_clim_min = fsno.groupby('time.month').mean().min(dim='month')

# Create masks
nonglc_mask = grid.PCT_GLC <= nonglc_pct_threshold
full_mask = nonglc_mask
# snow_mask = fsno_clim_min <= (snow_pct_threshold / 100)
# full_mask = snow_mask & nonglc_mask

# Load output
fhist = {}
fhist_full = {}
for v in variables:
    print(f'  {v}')
    name = '_'.join(v.split('_')[:-2])
    fhist_full[v] = xclim.load_fhist(v, keep_var_only=True)[name].sel(time=time_slice).reindex_like(grid, method='nearest', tolerance=1e-3)
    fhist_full[v] = fhist_full[v].where(full_mask)
    fhist_full[v].attrs['masks'] = f'gridcell percent glaciated land <= {nonglc_pct_threshold}'

    fhist[v] = fhist_full[v].chunk({'time': -1})

# Monthly PRECT (PRECC + PRECL)
print("  PRECT_month_1")
fhist["PRECT_month_1"] = fhist["PRECC_month_1"] + fhist["PRECL_month_1"]
fhist["PRECT_month_1"].attrs["long_name"] = "total monthly precipitation rate (PRECC + PRECL)"
fhist.pop("PRECC_month_1")
fhist.pop("PRECL_month_1")